# Session 3 — Advanced Conversational AI
**Date:** Sat 27 Jun 2026 | **Duration:** 4 hrs (9:30 am – 1:30 pm IST)  
**Course:** Python Applications with OpenAI — MSME Technology Development Center, Chennai

---

## Learning Objectives
By the end of this session you can:
- Apply prompt engineering patterns: zero-shot, few-shot, chain-of-thought
- Force the model to return valid, schema-conforming JSON using structured outputs
- Wire up function/tool calling: define tools, handle the two-step call pattern
- Stream tokens to the terminal as they arrive
- Wrap API calls with retry/error-handling for production robustness
- Make a basic call with the Responses API and explain how it differs from Chat Completions

## Agenda
| Time | Topic |
|------|-------|
| 00:00 | Recap of Session 2 + setup check |
| 00:15 | Prompt engineering patterns |
| 01:00 | Structured outputs (JSON mode & JSON Schema) |
| 01:40 | **BREAK** |
| 01:50 | Function / tool calling |
| 02:40 | Streaming responses |
| 03:00 | **BREAK** |
| 03:10 | Retries + error handling |
| 03:30 | First look at the Responses API |
| 03:50 | Exercises & recap |

In [1]:
# ── Session 3 Setup (run this cell first) ────────────────────────────────────
import os
import json
import sys
import time
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

_root = Path(os.getcwd())
for _ in range(5):
    if (_root / "utils").is_dir():
        break
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from utils.config import MODELS, get_client, estimate_cost
from utils.helpers import pretty_print, retry, mock_chat_response

MOCK  = os.getenv("MOCK", "0") == "1"
MODEL = MODELS["chat"]   # gpt-5.4-mini — swap to MODELS["frontier"] for gpt-5.5

if MOCK:
    print(f"MOCK mode ON — no real API calls. Model constant: {MODEL}")
else:
    try:
        client = get_client()
        client.models.list()
        print(f"Client is alive.  Default model: {MODEL}")
    except Exception as e:
        print(f"Setup issue: {e}\nCheck OPENAI_API_KEY in .env")

Client is alive.  Default model: gpt-5.4-mini


---
## 1. Prompt Engineering Patterns

Prompt engineering is the practice of crafting inputs that reliably produce the outputs you need. Three patterns are worth mastering before reaching for fine-tuning or RAG:

| Pattern | When to use it | How |
|---------|---------------|-----|
| **Zero-shot** | Simple, unambiguous tasks | Just ask directly |
| **Few-shot** | Tasks where format or style matters | Provide 2–5 input/output examples inside the prompt |
| **Chain-of-thought (CoT)** | Multi-step reasoning, maths, logic | Add `"Think step by step"` or ask for intermediate reasoning before the answer |

Additional techniques we'll use throughout the course:
- **Output format constraints** — specify the exact format you want (`"Reply with exactly one sentence."`, `"Return a JSON object with keys X, Y, Z."`)
- **Persona + constraints** — system prompt sets role; add rules as a numbered list for reliability
- **History trimming** — keep only the system message + last N turns to prevent runaway costs

In [ ]:
# function to conduct multiturn conversation
def multiturn_conversation(messages,user_question):

    'function to enable multi turn conversation by appending user query and response from model'

    messages.append({'role':'user',"content":user_question})

    response = client.chat.completions.create(
        messages = messages,
        model=MODEL
    )

    messages.append({'role':'assistant',"content":response.choices[0].message.content})


    return response

In [ ]:
# ── 1a. Zero-shot vs few-shot ────────────────────────────────────────────────
# TASK: Classify customer feedback as Positive / Negative / Neutral


query = "Wow, two days late. Truly impressive service."
#
# ZERO-SHOT — just ask:
messages = [
    {"role": "system", "content": "Classify the sentiment of customer feedback. Reply with exactly one word: Positive, Negative, or Neutral."},
    {"role": "user",   "content": query}
  ]

response = client.chat.completions.create(messages=messages,
                                          model=MODEL)

pretty_print(response)

#   Call API → print reply
#
# FEW-SHOT — add examples in the user message:
few_shot_prompt = f"""
  Classify sentiment as Positive, Negative, or Neutral.

  Examples:
  Input: "Product quality is outstanding, highly recommend!"
  Output: Positive

  Input: "Wrong item shipped, very disappointed."
  Output: Negative

  Input: "Arrived on time, packaging was standard."
  Output: Neutral

  Input: "Best damaged package I've ever received"
  Output: Negative

  Input: "{query}"
  Output:"""
#
messages = [{"role": "user", "content": few_shot_prompt}]


response = client.chat.completions.create(messages=messages,model=MODEL)

pretty_print(response)
#   Call API → print reply
#
# COMPARE: Are the results different? Which is more reliable?
# --- live-code below ---



────────────────────────────────────────────────────────────
Negative
────────────────────────────────────────────────────────────
  Tokens  prompt=41  completion=4  total=45
────────────────────────────────────────────────────────────
Negative
────────────────────────────────────────────────────────────
  Tokens  prompt=90  completion=4  total=94


In [6]:
# ── 1b. Chain-of-thought prompting ──────────────────────────────────────────
# TASK: A logic / reasoning problem where CoT dramatically improves accuracy
#

# WITHOUT CoT:
question = ("A shopkeeper buys 40 kg of rice at ₹45/kg and 60 kg at ₹50/kg. "
              "He sells the entire stock at ₹52/kg. What is his profit or loss?")
messages = [{"role": "user", "content": question}]

pretty_print(client.chat.completions.create(messages=messages,model=MODEL))
#   Call API at temperature=0 → print reply
#
# WITH CoT (add one phrase to the prompt):
cot_question = question + " Think step by step before giving the final answer."
messages = [{"role": "user", "content": cot_question}]
#   Call API at temperature=0 → print reply
#
# OBSERVE: Does adding "Think step by step" change the answer or the confidence?
# VERIFY manually: cost = 40*45 + 60*50 = 1800+3000 = 4800; revenue = 100*52 = 5200; profit = 400
# --- live-code below ---


pretty_print(client.chat.completions.create(messages=messages,model=MODEL))


────────────────────────────────────────────────────────────
Let’s calculate the total cost price and selling price.

### Cost Price (CP)
- 40 kg at ₹45/kg = ₹1800  
- 60 kg at ₹50/kg = ₹3000  

**Total CP = ₹1800 + ₹3000 = ₹4800**

### Selling Price (SP)
- Total rice = 40 kg + 60 kg = **100 kg**
- Sold at ₹52/kg = **₹5200**

### Profit
**Profit = SP - CP = ₹5200 - ₹4800 = ₹400**

### Answer:
**He makes a profit of ₹400.**
────────────────────────────────────────────────────────────
  Tokens  prompt=45  completion=130  total=175
────────────────────────────────────────────────────────────
Let’s calculate it step by step.

### 1) Cost price of rice
- 40 kg at ₹45/kg = **₹1800**
- 60 kg at ₹50/kg = **₹3000**

Total cost price = **₹1800 + ₹3000 = ₹4800**

### 2) Total quantity sold
- 40 kg + 60 kg = **100 kg**

### 3) Selling price
He sells all 100 kg at ₹52/kg:

Selling price = **100 × 52 = ₹5200**

### 4) Profit
Profit = Selling Price - Cost Price  
= **₹5200 - ₹4800 = ₹400**

### Final

In [ ]:
raw_prompt = """
A user wrote: "VPN connects but SAP hangs when I open reports. Restart didn't help."

What should we do?
"""






---
## 2. Structured Outputs

Asking for JSON in the prompt is unreliable — the model may add markdown fences, explanatory text, or invalid syntax. The API has two guaranteed ways to get valid JSON:

### Option A — JSON mode (`response_format={"type": "json_object"}`)
- Guarantees valid JSON is returned
- You define the expected schema **in the prompt** (the model infers the shape)
- Easy to add; no schema required

```python
response = client.chat.completions.create(
    model=MODEL, messages=messages,
    response_format={"type": "json_object"}
)
data = json.loads(response.choices[0].message.content)
```

### Option B — JSON Schema / Structured Outputs (`response_format={"type": "json_schema", ...}`)
- Guarantees the response matches **your exact schema** (field names, types, required fields)
- Use when downstream code depends on a fixed shape (databases, APIs, dashboards)
- Slightly more verbose to set up, far more reliable

> **Rule of thumb:** Use JSON mode for quick prototypes; use JSON Schema for anything that goes to production.

In [23]:
import json

In [24]:
# ── 2a. JSON mode — guaranteed valid JSON, schema in the prompt ──────────────
# TASK: Extract structured info from a plain-English invoice description
#
# STEPS:
#   1. invoice_text = (
#        "Invoice from Tata Steel Ltd dated 15-Jun-2026. "
#        "3 tonnes of hot-rolled coil at ₹62,000/tonne. "
#        "GST @ 18%. Payment due in 30 days."
#      )
#   2. messages = [
#        {"role": "system", "content": (
#            "Extract invoice details and return a JSON object with keys: "
#            "vendor, date, item, quantity, unit_price_inr, gst_percent, "
#            "subtotal_inr, gst_amount_inr, total_inr, payment_days."
#        )},
#        {"role": "user", "content": invoice_text}
#      ]
#   3. response = client.chat.completions.create(
#          model=MODEL, messages=messages,
#          response_format={"type": "json_object"}
#      )
#   4. data = json.loads(response.choices[0].message.content)
#      print(json.dumps(data, indent=2))
#   5. Access a field: print("Vendor:", data["vendor"])
# --- live-code below ---


invoice_text = (
       "Invoice from Tata Steel Ltd dated 15-Jun-2026. "
       "3 tonnes of hot-rolled coil at ₹62,000/tonne. "
       "GST @ 18%. Payment due in 30 days."
     )


messages = [
       {"role": "system", "content": (
           "Extract invoice details and return a JSON object with keys: "
           "vendor, date, item, quantity, unit_price_inr, gst_percent, "
           "subtotal_inr, gst_amount_inr, total_inr, payment_days."
       )},
       {"role": "user", "content": invoice_text}
     ]



response = client.chat.completions.create(messages=messages,
                               model=MODEL,
                               response_format={"type":"json_object"})


invoice_details = json.loads(response.choices[0].message.content)








In [25]:
invoice_details['quantity']

3

In [26]:
# ── 2b. JSON Schema (Structured Outputs) — enforce an exact shape ────────────
# TASK: Same invoice extraction but with a schema the API must strictly follow
#
# STEPS:
#   1. Define the schema dict:
schema = {
       "type": "object",
       "properties": {
         "vendor":          {"type": "string"},
         "date":            {"type": "string"},
         "item":            {"type": "string"},
         "quantity":        {"type": "number"},
         "unit_price_inr":  {"type": "number"},
         "gst_percent":     {"type": "number"},
         "total_inr":       {"type": "number"},
         "payment_days":    {"type": "integer"}
       },
       "required": ["vendor","date","item","quantity","unit_price_inr",
                    "gst_percent","total_inr","payment_days"],
       "additionalProperties": False
     }
#
#   2. 
response = client.chat.completions.create(
         model=MODEL,
         messages=[{"role": "user", "content": invoice_text}],
         response_format={
             "type": "json_schema",
             "json_schema": {"name": "invoice", "schema": schema, "strict": True}
         }
     )
#   3. data = json.loads(response.choices[0].message.content)
#      print(json.dumps(data, indent=2))
#   4. Try accessing a key that is NOT in the schema — what happens?
# --- live-code below ---

data = json.loads(response.choices[0].message.content)
print(json.dumps(data, indent=2))


{
  "vendor": "Tata Steel Ltd",
  "date": "15-Jun-2026",
  "item": "hot-rolled coil",
  "quantity": 3,
  "unit_price_inr": 62000,
  "gst_percent": 18,
  "total_inr": 218280,
  "payment_days": 30
}


> **Exercise 1 — Structured Product Catalogue Extractor**
>
> Given this unstructured product description:
> ```
> "Bajaj Pulsar NS200 2026 model. Engine: 199.5cc single cylinder. "
> "Power: 24.5 PS. Mileage: 35 km/l. Ex-showroom price Delhi: ₹1,42,000. "
> "Available in: Ebony Black, Pewter Grey, Burnt Red."
> ```
> 1. Design a JSON Schema with at least 6 fields (include an array field for colours).
> 2. Extract the data using `response_format={"type": "json_schema", ...}`.
> 3. Print `data["colours"]` — confirm it is a Python list.

In [ ]:
# your turn:


product_spec = ("Bajaj Pulsar NS200 2026 model. Engine: 199.5cc single cylinder. "
"Power: 24.5 PS. Mileage: 35 km/l. Ex-showroom price Delhi: ₹1,42,000. "
"Available in: Ebony Black, Pewter Grey, Burnt Red.")


---
## 3. Function / Tool Calling

Tool calling lets the model decide **when** to call a function and **what arguments** to pass. You provide the function definitions; the model emits a structured call; you execute it and feed the result back.

### The two-step pattern
```
Step 1:  You → API  (messages + tools list)
         API → You  (finish_reason="tool_calls", message.tool_calls=[...])

Step 2:  You execute the real function with the model's arguments
         You → API  (messages + tool result as role="tool" message)
         API → You  (finish_reason="stop", final text reply)
```

### Tool definition shape
```python
{
    "type": "function",
    "function": {
        "name": "function_name",
        "description": "What it does — the model reads this to decide when to call it.",
        "parameters": {
            "type": "object",
            "properties": {
                "param_name": {"type": "string", "description": "..."}
            },
            "required": ["param_name"]
        }
    }
}
```

In [49]:
# ── 3a. Define tools and make the first call ─────────────────────────────────
# TOOLS we will give the model:
#   1. get_usd_to_inr()         — returns current USD→INR exchange rate (mock: 83.5)
#   2. calculate_gst(amount, gst_rate) — returns GST-inclusive price
#
# STEPS:
#   1. Write two simple Python functions:
#        def get_usd_to_inr() -> float: return 83.5
#        def calculate_gst(amount: float, gst_rate: float) -> dict:
#            gst = round(amount * gst_rate / 100, 2)
#            return {"base": amount, "gst": gst, "total": round(amount + gst, 2)}
#
#   2. Define tools = [
#        {"type": "function", "function": {
#            "name": "get_usd_to_inr",
#            "description": "Returns the current USD to INR exchange rate.",
#            "parameters": {"type": "object", "properties": {}, "required": []}
#        }},
#        {"type": "function", "function": {
#            "name": "calculate_gst",
#            "description": "Calculate GST and total price for a base amount in INR.",
#            "parameters": {
#                "type": "object",
#                "properties": {
#                    "amount":   {"type": "number", "description": "Base price in INR"},
#                    "gst_rate": {"type": "number", "description": "GST rate as percent, e.g. 18"}
#                },
#                "required": ["amount", "gst_rate"]
#            }
#        }}
#      ]
#
#   3. messages = [{"role": "user", "content":
#        "A software subscription costs $120. What is the price in INR including 18% GST?"}]
#      response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
#      print("finish_reason:", response.choices[0].finish_reason)   # expect "tool_calls"
#      print("tool_calls:",    response.choices[0].message.tool_calls)
# --- live-code below ---


def get_usd_to_inr():

    'returns the current usd to inr conversion rate'

    return 96.2


def calculate_gst(amount,gst_rate):

    gst = round(amount * gst_rate / 100, 2)
    
    return {"base": amount, "gst": gst, "total": round(amount + gst, 2)}


tools = [
       {"type": "function", "function": {
           "name": "get_usd_to_inr",
           "description": "Returns the current USD to INR exchange rate.",
           "parameters": {"type": "object", "properties": {}, "required": []}
       }},
       {"type": "function", "function": {
           "name": "calculate_gst",
           "description": "Calculate GST and total price for a base amount in INR.",
           "parameters": {
               "type": "object",
               "properties": {
                   "amount":   {"type": "number", "description": "Base price in INR"},
                   "gst_rate": {"type": "number", "description": "GST rate as percent, e.g. 18"}
               },
               "required": ["amount", "gst_rate"]
           }
       }}
     ]


messages = [{"role": "user", 
"content":"A software subscription costs $120. What is the price in INR including 18% GST?"}]


response = client.chat.completions.create(messages=messages,model=MODEL,tools=tools)


In [50]:
response.choices[0].message.tool_calls[0]

ChatCompletionMessageFunctionToolCall(id='call_jx0UU1xCgbczst7wg425VCjL', function=Function(arguments='{}', name='get_usd_to_inr'), type='function')

In [59]:
tool_called = response.choices[0].message.tool_calls[0].function.name

In [60]:
tool_called

'calculate_gst'

In [61]:
input_args = json.loads(response.choices[0].message.tool_calls[0].function.arguments)

In [62]:
input_args

{'amount': 11544, 'gst_rate': 18}

In [63]:
if tool_called == 'get_usd_to_inr':

    print(get_usd_to_inr())

elif tool_called == 'calculate_gst':

    print(calculate_gst(amount=input_args['amount'],gst_rate=input_args['gst_rate']))

{'base': 11544, 'gst': 2077.92, 'total': 13621.92}


In [56]:
messages.append(response.choices[0].message) 

messages.append({"role":"tool",
               "tool_call_id":"call_jx0UU1xCgbczst7wg425VCjL",
               "content":str(96.2)})

In [57]:
response = client.chat.completions.create(messages=messages,model=MODEL,tools=tools)

In [58]:
response.choices[0].message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_O01enimtI82HV1noACjWIC7W', function=Function(arguments='{"amount":11544,"gst_rate":18}', name='calculate_gst'), type='function')])

In [ ]:
# ── 3b. Handle the tool call — execute and return the result ─────────────────
# STEPS (continuing from 3a, with the response object in hand):
#
#   1. tool_call = response.choices[0].message.tool_calls[0]
#      fn_name = tool_call.function.name
#      fn_args = json.loads(tool_call.function.arguments)
#      print(f"Model wants to call: {fn_name}({fn_args})")
#
#   2. Dispatch to the real function:
#      fn_map = {"get_usd_to_inr": get_usd_to_inr, "calculate_gst": calculate_gst}
#      result = fn_map[fn_name](**fn_args)
#      print("Function result:", result)
#
#   3. Append both the assistant's tool-call message AND the tool result:
#      messages.append(response.choices[0].message)        # assistant's tool_calls message
#      messages.append({
#          "role":         "tool",
#          "tool_call_id": tool_call.id,
#          "content":      json.dumps(result)
#      })
#      print("Updated messages:", len(messages), "entries")
# --- live-code below ---


In [ ]:
# ── 3c. Second API call — model reads the result and gives a final answer ─────
# NOTE: The model may call another tool (multi-hop) or give the final text reply.
#       In production wrap this in a loop; for now we handle one extra hop manually.
#
# STEPS:
#   1. final = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
#      print("finish_reason:", final.choices[0].finish_reason)  # may still be "tool_calls"
#
#   2. If finish_reason == "tool_calls": handle the second tool call (same pattern as 3b)
#      Then call the API one more time to get the text reply.
#
#   3. Print the final answer:
#      print(final.choices[0].message.content)
#      # Expected: "The software subscription costs ₹10,020 including 18% GST"
#
#   4. pretty_print(final)
# --- live-code below ---


> **Exercise 2 — Add a Third Tool: Convert Units**
>
> Add a tool `convert_weight(value, from_unit, to_unit)` that converts between `"kg"`, `"tonne"`, and `"pound"`.
> 1. Implement the Python function.
> 2. Add it to the `tools` list.
> 3. Ask: *"I have 2.5 tonnes of steel. What is that in pounds, and what is the GST-inclusive price at ₹62,000/tonne (18% GST)?"*
> 4. Trace how many tool calls the model makes before giving the final answer.

In [ ]:
# your turn:


---
## 4. Streaming Responses

By default the API waits until the full response is ready before returning it. With `stream=True` you get an **iterator of chunks** — each chunk carries a small delta of text. This makes long responses feel instant and lets you update the UI as the model types.

```python
stream = client.chat.completions.create(
    model=MODEL, messages=messages, stream=True
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()  # final newline
```

> **Gotcha:** `response.usage` is not available on individual chunks. If you need token counts with streaming, pass `stream_options={"include_usage": True}` — the last chunk will carry a `usage` object.

In [ ]:
# ── 4a. Stream tokens to the terminal ───────────────────────────────────────
# GOAL: Feel the difference between batch and streaming responses
#
# STEPS:
#   1. messages = [{"role": "user", "content":
#        "Write a short paragraph about the history of the Indian textile industry."}]
#
#   2. NON-STREAMING (for comparison — notice the delay before output):
#      response = client.chat.completions.create(model=MODEL, messages=messages)
#      print(response.choices[0].message.content)
#
#   3. STREAMING (tokens appear as they are generated):
#      print("\n--- Streaming version ---")
#      stream = client.chat.completions.create(
#          model=MODEL, messages=messages,
#          stream=True,
#          stream_options={"include_usage": True}   # get usage on the last chunk
#      )
#      full_text = ""
#      for chunk in stream:
#          delta = chunk.choices[0].delta.content
#          if delta:
#              print(delta, end="", flush=True)
#              full_text += delta
#          if chunk.usage:                          # last chunk
#              print(f"\n\nTokens: {chunk.usage.total_tokens}")
# --- live-code below ---


---
## 5. Retries and Error Handling

API calls can fail transiently. Common exceptions from the `openai` SDK:

| Exception | Cause | Action |
|-----------|-------|---------|
| `openai.AuthenticationError` | Wrong or missing API key | Fix key; don't retry |
| `openai.RateLimitError` | Too many requests per minute | Retry with exponential backoff |
| `openai.APIConnectionError` | Network issue | Retry 2–3 times |
| `openai.BadRequestError` | Invalid input (e.g. `temperature > 2`) | Fix the request; don't retry |
| `openai.APIStatusError` | 5xx server errors | Retry with backoff |

The `retry()` helper in `utils/helpers.py` provides a simple fixed-delay retry. For production you'd use exponential backoff — we'll build a version here.

In [ ]:
# ── 5a. Robust API call with typed exception handling ───────────────────────
# GOAL: Catch specific errors, retry transient ones, surface clean messages
#
# STEPS:
#   1. import openai
#
#   2. def safe_chat(messages, model=MODEL, retries=3, backoff=2.0):
#        for attempt in range(1, retries + 1):
#            try:
#                return client.chat.completions.create(model=model, messages=messages)
#            except openai.AuthenticationError as e:
#                print(f"Auth error — check your API key: {e}"); raise
#            except openai.BadRequestError as e:
#                print(f"Bad request — fix the input: {e}"); raise
#            except (openai.RateLimitError, openai.APIConnectionError,
#                    openai.APIStatusError) as e:
#                if attempt == retries: raise
#                wait = backoff ** attempt
#                print(f"Attempt {attempt}/{retries} failed ({type(e).__name__}) — "
#                      f"retrying in {wait:.1f}s")
#                time.sleep(wait)
#
#   3. Test with a valid call:
#        response = safe_chat([{"role": "user", "content": "Say hello in Hindi."}])
#        pretty_print(response)
#
#   4. Simulate a failure: temporarily set client = OpenAI(api_key="bad-key")
#        try: safe_chat([{"role": "user", "content": "hello"}])
#        except openai.AuthenticationError: print("Caught auth error cleanly")
#      then restore the real client
# --- live-code below ---


---
## 6. First Look at the Responses API

> **Why a new API?** Chat Completions was designed for stateless request/response. The **Responses API** is designed for *agentic* workflows: it manages conversation state server-side, natively supports built-in tools (web search, file search, code interpreter, image generation), and composes multi-step tasks in a single call. We'll go deep on tools in Session 5; here we just compare the interfaces side by side.

### Chat Completions vs Responses API
| | Chat Completions | Responses API |
|-|-----------------|---------------|
| Endpoint method | `client.chat.completions.create()` | `client.responses.create()` |
| Input param | `messages=[...]` | `input="..."` (string or list) |
| Output text | `response.choices[0].message.content` | `response.output_text` |
| State management | You maintain `messages` list | Server-side via `previous_response_id` |
| Built-in tools | Only function calling | `web_search`, `file_search`, `code_interpreter`, `image_generation` |
| Best for | Fundamentals, chat apps | Agents, RAG, multi-modal pipelines |

**When to use which:**  
Sessions 1–2 content → Chat Completions.  
Sessions 4–6 content (tools, RAG, agents) → Responses API.

In [ ]:
# ── 6a. Basic Responses API call ────────────────────────────────────────────
# GOAL: Make the simplest possible Responses API call and compare the interface
#
# STEPS:
#   1. prompt = "Name the three most important Python libraries for data science. "
#               "Reply in one sentence."
#
#   2. CHAT COMPLETIONS way (familiar):
#      r1 = client.chat.completions.create(
#          model=MODEL,
#          messages=[{"role": "user", "content": prompt}]
#      )
#      print("Chat Completions:", r1.choices[0].message.content)
#
#   3. RESPONSES API way (new):
#      r2 = client.responses.create(
#          model=MODEL,
#          input=prompt
#      )
#      print("Responses API:   ", r2.output_text)
#
#   4. Compare: print(type(r1), type(r2)) — different response object shapes
# --- live-code below ---


In [ ]:
# ── 6b. Responses API — server-side conversation state ──────────────────────
# GOAL: Show how previous_response_id replaces the manual messages list
#
# STEPS:
#   1. Turn 1:
#      r1 = client.responses.create(
#          model=MODEL,
#          input="My name is Arjun and I work in logistics in Chennai."
#      )
#      print("Turn 1:", r1.output_text)
#      print("Response ID:", r1.id)   # we'll use this to continue
#
#   2. Turn 2 — no messages list needed, just reference the previous response:
#      r2 = client.responses.create(
#          model=MODEL,
#          input="What's my name and city?",
#          previous_response_id=r1.id   # the API looks up the full context
#      )
#      print("Turn 2:", r2.output_text)
#      # Model should recall "Arjun" and "Chennai" without you sending history
#
#   3. Discuss the trade-off: server-side state is convenient but you lose
#      direct control over what's in context (use cases for each approach)
# --- live-code below ---


> **Exercise 3 — Structured + Tool Calling Pipeline**
>
> Combine what you have learned:
> 1. Build a tool `lookup_product(product_name)` that returns a mock product dict  
>    `{"name": ..., "price_inr": ..., "gst_rate": 18}`.
> 2. Ask: *"What is the GST-inclusive price of a Bajaj Pulsar NS200?"*  
>    The model should call `lookup_product`, then `calculate_gst`, then give you a final answer.
> 3. Force the final answer into a JSON Schema with fields `product`, `base_price`, `gst`, `total`.
>    Hint: use `response_format` in the second (or third) API call once you have the numbers.

In [ ]:
# your turn:


---
## Common Pitfalls

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Using JSON mode without mentioning JSON in the prompt | `BadRequestError` — the model must be told to produce JSON | Add `"Respond with a JSON object."` to the system or user message |
| `additionalProperties` not set to `False` in JSON Schema | Model adds unexpected fields that break downstream parsing | Always set `"additionalProperties": False` in strict schemas |
| Forgetting to append `response.choices[0].message` before the tool result | `BadRequestError` — tool result has no matching `tool_call_id` context | Append the assistant's full message object first, then the `role=\"tool\"` message |
| Calling `tool_call.function.arguments` as a dict | `AttributeError` — it is a JSON **string** | Always `json.loads(tool_call.function.arguments)` |
| Streaming + accessing `response.usage` | `AttributeError` — usage not on stream object | Pass `stream_options={"include_usage": True}` and read from the last chunk |
| Retrying `AuthenticationError` | Wastes time, logs errors | Catch auth errors separately; only retry transient network/rate errors |

---
## Recap

- **Few-shot** beats zero-shot when format matters; **chain-of-thought** beats both for reasoning tasks.
- `response_format={"type": "json_object"}` guarantees valid JSON; `json_schema` with `strict=True` guarantees a specific shape — prefer the latter for production.
- Tool calling is a **two-step pattern**: (1) model emits `tool_calls`, you execute and return results; (2) model reads results and gives a final text reply.
- `stream=True` returns tokens as an iterator; add `stream_options={"include_usage": True}` to still get token counts.
- Catch exceptions by type: retry `RateLimitError` / `APIConnectionError` with backoff; surface `AuthenticationError` immediately.
- The Responses API replaces the manual `messages` list with server-side state via `previous_response_id` — it will be our primary interface in Sessions 4–6.

---
## Homework / Capstone Increment

Before Session 4 (Sun 28 Jun):

1. **Prompt patterns:** Apply few-shot prompting to your domain (`MY_SYSTEM`). Provide 3 examples where zero-shot gives a wrong or imprecise answer and few-shot fixes it. Save the examples as a variable `FEW_SHOT_EXAMPLES`.

2. **Structured output:** Design a JSON Schema for the core data entity in your domain (e.g., a patient record, a purchase order, a logistics shipment). Extract an instance from a plain-English description using `response_format`.

3. **Tool:** Add one real or mock tool relevant to your domain to `apps/chat_app.py`. Wire up the two-step pattern so the Streamlit app can answer questions that need the tool.

4. **Streaming in the UI:** Update `apps/chat_app.py` to stream the assistant's response using `st.write_stream()` (Streamlit 1.31+) or a manual `st.empty()` + update loop. Compare the UX feel.

5. **Optional:** Replace the `messages`-based history in `apps/chat_app.py` with the Responses API (`previous_response_id`). Note any behaviour differences.